# Where the data comes from

Cristiano Ronaldo was missing from this dataset for weeks and I had no idea.

His statistics were there the whole time. What was missing was his *identity*,
so nothing could be pooled into a career, and when I finally ran a check against
award voting it reported the five-time Ballon d'Or winner as a player it had
never heard of.

Finding that took an afternoon. Building the machinery that surfaced it took a
lot longer, and that machinery is what this post is about: how twenty-five
seasons of football actually arrive on my laptop, and what state they are in when
they get here.

Nothing in this chapter fetches anything. Every cell runs off files committed to
the repository, so you can follow along.

## There is no download button

I want to start here because it shapes every decision that comes after.

FBref has no API. There is no bulk export. And there is Cloudflare sitting in
front of the whole thing. Those three facts have consequences:

| What is true | What it forces |
|---|---|
| No API | Every number gets parsed out of an HTML page |
| Cloudflare | You need a real browser, so `soccerdata` drives undetected Chrome |
| Redistribution is restricted | Raw pages stay on my machine. Only derived aggregates get published |
| Roughly 30 seconds a page, 625 pages | A cold build takes about five hours |

That last row is the one that dictated the architecture. If refetching is
expensive, then **iterating on the definition must never refetch.** That is the
whole design in one sentence.

So the network lives in exactly one stage and nowhere else:

```
scrape ──▶ vault/raw ──▶ clean ──▶ vault/clean ──▶ rank ──▶ vault/derive ──▶ data/sample
(network)   (cached)     (identity)  (validated)    (stats)    (outputs)      (committed)
```

Everything to the right of that first arrow reads Parquet. When I change my mind
about what greatness means, the whole chain reruns in about five minutes, with
the wifi off.

## What one page actually looks like

Here is a real slice of FBref's Premier League 2000-01 table, saved as a test
fixture. Forty rows of what all 625 pages look like.

In [1]:
import pandas as pd

from gambeta import laws
from gambeta.scouts.fbref import flatten

raw = pd.read_pickle("../tests/fixtures/fbref_raw.pkl")

print(f"shape          {raw.shape}")
print(f"index levels   {raw.index.names}")
print(f"column levels  {raw.columns.nlevels}")
raw.iloc[:4, :6]

shape          (40, 21)
index levels   ['league', 'season', 'team', 'player']
column levels  2


nation pos age  born  \
                                                                         
league             season team    player                                 
ENG-Premier League 0001   Arsenal Alex Manninger     AUT  GK  23  1977   
                                  Ashley Cole        ENG  DF  19  1980   
                                  David Seaman       ENG  GK  36  1963   
                                  Dennis Bergkamp    NED  FW  31  1969   

                                                  Playing Time         
                                                            MP Starts  
league             season team    player                               
ENG-Premier League 0001   Arsenal Alex Manninger            11     11  
                                  Ashley Cole               17     15  
                                  David Seaman              24     24  
                                  Dennis Bergkamp           25     19

Two things about that frame will bite you, and neither is anybody's fault
here. It is just how the source publishes.

**The columns are two levels deep.** You get `("Playing Time", "MP")`, not `mp`.
If you select a column off this frame you get a two-level result back, which then
refuses to merge against anything flat and hands you
`MergeError: Not allowed to merge between different levels`. I lost time to that
one. Every reader in this project now flattens headers *before* selecting
anything at all.

**The keys are in the index, not the columns.** `league`, `season`, `team` and
`player` are what identify a row, and not one of them is a column you can join
on.

`flatten` sorts out both, and renames the twenty or so columns I actually use.

In [2]:
flat = flatten(raw)

print(f"{raw.shape} -> {flat.shape}\n")
print("kept:", ", ".join(flat.columns[:12]), "...")
flat.head(3)[["season", "team", "player", "born", "mp", "starts", "minutes", "goals"]]

(40, 21) -> (40, 18)

kept: league, season, team, player, nation, pos, age, born, mp, starts, minutes, goals ...


,season,team,player,born,mp,starts,minutes,goals
0,0001,Arsenal,Alex Manninger,1977.0,11,11,990,0
1,0001,Arsenal,Ashley Cole,1980.0,17,15,1300,3
2,0001,Arsenal,David Seaman,1963.0,24,24,2160,0


Notice what is *not* in there. FBref publishes `Gls/90`, `Ast/90` and `G+A/90`
already calculated, and I throw every one of them away.

That looks wasteful until you think about the denominator. A per-90 that arrives
precomputed is a per-90 whose denominator you cannot see, and this project sums a
transferred player's minutes across two clubs before dividing. If I kept both
columns I would have two numbers answering the same question and disagreeing
with each other, which is worse than having one.

## Why the cache is broken into small pieces

The scrape writes **one HTML file per (league, season, table)**. That is 634
files and 1.19 GB. Wikidata gets cached as **one Parquet per birth year**, with
the query text hashed into the filename.

I did try the obvious thing first, which is caching the finished artifact, and I
can tell you exactly what it cost. Two of forty-one birth-year cohorts failed to
fetch. Repairing them meant rerunning all forty-one: about thirty-five minutes to
recover four minutes of real work. Per unit, the same repair now costs one
request.

The query hash is there for a subtler reason, and it is the rule I would tattoo
on someone: **a cached answer to a different question is not a saving, it is a
wrong answer.** Change the SPARQL and the filename changes with it, so the old
file can never be served up for the new query.

In [3]:
# The naming scheme *is* the cache key. No index, no manifest of manifests.
for name in [
    "vault/raw/FBref/players_ENG-Premier League_0001_standard.html",
    "vault/raw/FBref/players_ITA-Serie A_1718_misc.html",
    "vault/raw/wikidata/crosswalk_c2990366_1987.parquet",
]:
    stem = name.rsplit("/", 1)[1]
    print(f"  {stem}")

print("\n  634 FBref pages (1.19 GB) + 41 Wikidata cohorts (7.1 MB)")
print("  c2990366 is a fingerprint of the query text: edit the SPARQL, miss the cache.")

  players_ENG-Premier League_0001_standard.html
  players_ITA-Serie A_1718_misc.html
  crosswalk_c2990366_1987.parquet

  634 FBref pages (1.19 GB) + 41 Wikidata cohorts (7.1 MB)
  c2990366 is a fingerprint of the query text: edit the SPARQL, miss the cache.


## Storage: Parquet, a schema, and a receipt

Every write goes through one function, and it does three things in a fixed
order. **Validate, then write, then record.**

The order matters. Validation happens before any file gets created, so a
rejected frame never leaves a half-written artifact lying around. A partial
Parquet that looks complete is a lot worse than no Parquet at all, because you
will trust it.

In [4]:
import json
import tempfile
from pathlib import Path

from gambeta import locker

with tempfile.TemporaryDirectory() as tmp:
    path = locker.write(flat, Path(tmp) / "demo.parquet", laws.PLAYER_SEASON_RAW, source="fbref")
    receipt = json.loads(locker.manifest_path(path).read_text())

print(json.dumps({k: v for k, v in receipt.items() if k != "columns"}, indent=2))
print(f"columns: {len(receipt['columns'])}")

[08/12/26 02:29:43] INFO     No custom team name replacements found. You can configure these in       _config.py:91
                             C:\Users\Aditya_Mishra\soccerdata\config\teamname_replacements.json.                  

                    INFO     No custom league dict found. You can configure additional leagues in    _config.py:189
                             C:\Users\Aditya_Mishra\soccerdata\config\league_dict.json.                            

{
  "source": "fbref",
  "written_at": "2026-08-11T20:59:43.235754+00:00",
  "rows": 40,
  "soccerdata_version": "1.9.1"
}

columns: 18

The manifest earns its keep by making a **partial** scrape tell you it is
partial. A Parquet file with 3,000 rows looks exactly the same whether that is
all the data there was or the fraction that made it in before a browser process
died. The row count and the timestamp are the only things that will tell you
which one you are holding.

The schema does the other half of the job. It is a contract, not documentation.

In [5]:
broken = flat.assign(minutes=flat["minutes"] * -1)

try:
    laws.PLAYER_SEASON_RAW.validate(broken)
except laws.ValidationError as exc:
    print(str(exc).split("\n")[0])
    print("\nNo file was written. That is the whole point of validating first.")

Column 'minutes' failed element-wise validator number 0: greater_than_or_equal_to(0) failure cases: -990, -1300, -2160, -1692, -172, -2202, -2385, -810, -270, -1223, -2382, -2520, -16, -421, -1330, -1395, -43, -2552, -2486, -2489, -1198, -1729, -3420, -2825, -895, -608, -2363, -2290, -1122, -704, -2417, -1875, -259, -2546, -2325, -1586, -1369, -7, -527, -499


No file was written. That is the whole point of validating first.

## Why Parquet and not CSV

Parquet is columnar, compressed and typed. It is the third one I care about, and
a single column makes the case better than I can.

Seasons are labelled `"0001"` for 2000-01 and `"2425"` for 2024-25. Four
characters, stored as a string, deliberately, so they sort properly and so
nobody can accidentally do arithmetic to them.

Watch what a round trip through CSV does to that.

In [6]:
import io

buffer = io.StringIO()
flat.to_csv(buffer, index=False)
roundtrip = pd.read_csv(io.StringIO(buffer.getvalue()))

print(f"in memory    season = {flat['season'].iloc[0]!r}  ({flat['season'].dtype})")
print(f"through CSV  season = {roundtrip['season'].iloc[0]!r}  ({roundtrip['season'].dtype})")
print(f"\n2000-01 came back as the integer {roundtrip['season'].iloc[0]}.")
print("Parquet stores the dtype, so this cannot happen.")

in memory    season = '0001'  (str)

through CSV  season = np.int64(1)  (int64)


2000-01 came back as the integer 1.

Parquet stores the dtype, so this cannot happen.

CSV has no types. Every read is a guess, the guess depends on which values
happen to be in the file, and a file that parses correctly today can parse
differently tomorrow because one new row changed what pandas inferred.

## The four layers, and what each one is allowed to do

| Layer | Written by | The rule |
|---|---|---|
| `vault/raw` | the scrape | Never edited by hand. The only stage that touches the network |
| `vault/clean` | identity and collapsing | One row per player-season. Schema-validated |
| `vault/derive` | the statistics | Requirements, offsets, rankings |
| `data/sample` | publication | 5 MB or less, committed to git, enforced by pre-commit |

`data/sample` is the layer this book reads. Every figure in every chapter comes
out of it, and that is why the whole thing renders with no network connection and
no copy of Chrome.

In [7]:
SAMPLE = "../data/sample"
published = pd.read_parquet(f"{SAMPLE}/player_season_scored.parquet")

megabytes = published.memory_usage(deep=True).sum() / 1e6
print(f"{len(published):,} player-seasons published, {megabytes:.1f} MB in memory")
print(f"leagues: {', '.join(sorted(published['league'].unique()))}")
print(f"seasons: {published['season'].min()} to {published['season'].max()}")

39,877 player-seasons published, 13.8 MB in memory

leagues: ENG-Premier League, ESP-La Liga, FRA-Ligue 1, GER-Bundesliga, ITA-Serie A

seasons: 0001 to 2425

## How it breaks

**The scrape runs one league at a time, and I learned that the hard way.**

`soccerdata` spawns something like forty Chrome processes per session. I tried
three leagues at once. My 16 GB machine went from 15 GB free to 0.8 GB, and each
fetch slowed from 30 seconds to 250. Running three in parallel was more than
eight times slower *per page* than running one. The obvious parallelisation is a
pessimisation here, and the only way to know that is to measure it.

**A cache will happily hide a failed fetch from you.** A page that comes back as
a Cloudflare challenge instead of a table still gets written to disk. It is a
file, it has a perfectly sensible name, and the next run finds it and is
satisfied. This is why the manifest records row counts, and why I check the
season-by-season coverage of every table instead of assuming it. From a
directory listing, an empty table and a missing table look identical.

**Two of the four bugs in this project's history got in right here.** One was a
query that anchored on the wrong Wikidata property and quietly threw away 56% of
the candidate players. The other was a join key that was not unique. Neither one
raised an exception. Both get their own chapter.